In [ ]:
import os
import tifffile as tif
import numpy as np
import cv2
from sklearn.model_selection import train_test_split


# --------------------------------------------------
# PATHS
# --------------------------------------------------

main_folderpath = r'Y:\coskun-lab\Ivan\Aditi Pix2pix\data_v2'

folder_paths = {
    "sample_1": r'Y:/coskun-lab/Aditi/01/Results/registration/Multiplex 2/ANCA 9/1x',
    "sample_2": r'Y:/coskun-lab/Aditi/01/Results/registration/Multiplex 2/CTRL 7/1x',
    "sample_3": r'Y:/coskun-lab/Aditi/01/Results/registration/Multiplex 8/Med 2/1x/',
    "sample_4": r'Y:/coskun-lab/Aditi/01/Results/registration/Multiplex 9/L554-4/1x'
}


# --------------------------------------------------
# CONFIG
# --------------------------------------------------

PATCH_SIZE = 512
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_STATE = 42


# --------------------------------------------------
# CREATE PIX2PIX FOLDERS
# --------------------------------------------------

def create_pix2pix_folders(base_folder):

    for split in ["train","val","test"]:
        os.makedirs(os.path.join(base_folder,"A",split),exist_ok=True)
        os.makedirs(os.path.join(base_folder,"B",split),exist_ok=True)


# --------------------------------------------------
# FAST TILING
# --------------------------------------------------

def tile_image(img,patch_size):

    H,W = img.shape[:2]

    H_trim = (H//patch_size)*patch_size
    W_trim = (W//patch_size)*patch_size

    img = img[:H_trim,:W_trim]

    C = img.shape[2]

    patches = img.reshape(
        H_trim//patch_size,patch_size,
        W_trim//patch_size,patch_size,
        C
    ).swapaxes(1,2)

    patches = patches.reshape(-1,patch_size,patch_size,C)

    return patches


# --------------------------------------------------
# SPLIT
# --------------------------------------------------

def split_indices(n):

    idx = np.arange(n)

    train_idx,temp_idx = train_test_split(
        idx,
        test_size=(1-TRAIN_RATIO),
        random_state=RANDOM_STATE,
        shuffle=True
    )

    val_ratio_adjusted = VAL_RATIO/(VAL_RATIO+TEST_RATIO)

    val_idx,test_idx = train_test_split(
        temp_idx,
        test_size=(1-val_ratio_adjusted),
        random_state=RANDOM_STATE,
        shuffle=True
    )

    return train_idx,val_idx,test_idx


# --------------------------------------------------
# SAVE PNG
# --------------------------------------------------

def save_png(img,path):

    os.makedirs(os.path.dirname(path),exist_ok=True)

    if img.dtype != np.uint8:
        img = img.astype(np.float32)
        img = (img-img.min())/(img.max()-img.min()+1e-8)
        img = (img*255).astype(np.uint8)

    img = cv2.cvtColor(img,cv2.COLOR_RGB2BGR)

    cv2.imwrite(path,img)


# --------------------------------------------------
# BUILD DATASET FOR ONE SAMPLE
# --------------------------------------------------

def process_sample(sample_name):

    sample_folder = folder_paths[sample_name]

    print(f"\nProcessing {sample_name}")

    HE = tif.imread(os.path.join(sample_folder,'HE_registered.tif'))
    PAS = tif.imread(os.path.join(sample_folder,'PAS_registered.tif'))
    TRI = tif.imread(os.path.join(sample_folder,'Trichrome_registered.tif'))

    print("Images loaded")

    HE_patches = tile_image(HE,PATCH_SIZE)
    PAS_patches = tile_image(PAS,PATCH_SIZE)
    TRI_patches = tile_image(TRI,PATCH_SIZE)

    print("Number of patches:",len(HE_patches))

    train_idx,val_idx,test_idx = split_indices(len(HE_patches))

    splits = {
        "train":train_idx,
        "val":val_idx,
        "test":test_idx
    }

    pas_folder = os.path.join(main_folderpath,f"{sample_name}_PAS")
    tri_folder = os.path.join(main_folderpath,f"{sample_name}_TRI")

    create_pix2pix_folders(pas_folder)
    create_pix2pix_folders(tri_folder)

    for split_name,indices in splits.items():

        print("Saving",split_name)

        for new_i,old_i in enumerate(indices):

            he = HE_patches[old_i]
            pas = PAS_patches[old_i]
            tri = TRI_patches[old_i]

            # HE → PAS
            save_png(
                he,
                os.path.join(pas_folder,"A",split_name,f"{new_i}.png")
            )

            save_png(
                pas,
                os.path.join(pas_folder,"B",split_name,f"{new_i}.png")
            )

            # HE → TRI
            save_png(
                he,
                os.path.join(tri_folder,"A",split_name,f"{new_i}.png")
            )

            save_png(
                tri,
                os.path.join(tri_folder,"B",split_name,f"{new_i}.png")
            )

    print(f"{sample_name} done.")


# --------------------------------------------------
# RUN ALL SAMPLES
# --------------------------------------------------

# for sample in folder_paths:
#     process_sample(sample)

In [ ]:
process_sample('sample_2')

In [ ]:
for sample in folder_paths:
    try:
        process_sample(sample)
    except Exception as e:
        print(f"Skipping {sample} due to error: {e}")